In [1]:
from marearts_crystal import ma_crystal

# Initialize
mask = ma_crystal("your-secret-key")

# Encrypt data
data = b"Sensitive information"
encrypted = mask.encrypt_data(data)
decrypted = mask.decrypt_data(encrypted)

# Generate serial key
serial_key = mask.generate_serial_key("user@email.com", "2025-09-10", "2025-12-31")
valid = mask.validate_serial_key("user@email.com", serial_key)
print(valid, serial_key)

('2025-09-10', '2025-12-31') MAEV2:gAAAAABowsP1ZYbJfvF94Q_YpGKrf28gWq0IIohNT169Qgy4EK7dY0ZV6KcfIeJzykFbB7stspH6IlZxU2uCF36LF8ayW6M8wyS-KQvJRtoajzOlQL8xHjA=


In [2]:
from marearts_crystal import ma_crystal

mask = ma_crystal("your-company-secret")

# Generate license key
today = mask.get_today_date()
expire = mask.generate_end_date(years=1)  # 1 year license
license_key = mask.generate_serial_key("customer@email.com", today, expire)

print(f"License Key: {license_key}")

# Validate license
result = mask.validate_serial_key("customer@email.com", license_key)
if result:
    start_date, end_date = result
    if mask.validate_date(start_date, end_date):
        print("✅ License is active")
    else:
        print("❌ License expired")

License Key: MAEV2:gAAAAABowsP2x5ef_QCr_GTES6E1LP4Gc_1otk9G0JbjdfqtXKFK8dThm90HYVglxN7TPvKSWnZqCT11q_a6xaFZAcVeuAzJu_du4zJ2ko75gHucKal15Zw=
✅ License is active


In [3]:
# Encrypt a file
with open("screen.png", "rb") as f:
    file_data = f.read()

encrypted = mask.encrypt_data(file_data)

with open("screen.png.enc", "wb") as f:
    f.write(encrypted)

# Decrypt a file
with open("screen.png.enc", "rb") as f:
    encrypted_data = f.read()

decrypted = mask.decrypt_data(encrypted_data)

with open("screen_decrypted.png", "wb") as f:
    f.write(decrypted)

In [4]:
import json
from marearts_crystal import ma_crystal

mask = ma_crystal("app-secret-key")

# Save encrypted config
config = {
    "api_key": "secret-api-key",
    "database": "postgresql://localhost/mydb",
    "debug": False
}

encrypted_config = mask.encrypt_string(json.dumps(config))
with open("config.enc", "w") as f:
    f.write(encrypted_config)

# Load encrypted config
with open("config.enc", "r") as f:
    encrypted = f.read()

decrypted = mask.decrypt_string(encrypted)
config = json.loads(decrypted)
print(config)


{'api_key': 'secret-api-key', 'database': 'postgresql://localhost/mydb', 'debug': False}


In [5]:
from marearts_crystal import ma_crystal
from datetime import datetime

class LicenseManager:
    def __init__(self, secret_key):
        self.mask = ma_crystal(secret_key)
    
    def create_license(self, email, license_type="standard"):
        today = self.mask.get_today_date()
        
        if license_type == "trial":
            end_date = self.mask.generate_end_date(days=30)
        elif license_type == "standard":
            end_date = self.mask.generate_end_date(years=1)
        elif license_type == "premium":
            end_date = self.mask.generate_end_date(years=3)
        else:  # lifetime
            end_date = "2099-12-31"
        
        return self.mask.generate_serial_key(email, today, end_date)
    
    def verify_license(self, email, license_key):
        result = self.mask.validate_serial_key(email, license_key)
        if not result:
            return False, "Invalid license key"
        
        start_date, end_date = result
        if self.mask.validate_date(start_date, end_date):
            days_left = (datetime.strptime(end_date, "%Y-%m-%d") - datetime.now()).days
            return True, f"Valid until {end_date} ({days_left} days remaining)"
        else:
            return False, f"License expired on {end_date}"

# Usage
lm = LicenseManager("company-secret-2024")

# Create licenses
trial_key = lm.create_license("trial@user.com", "trial")
premium_key = lm.create_license("premium@user.com", "premium")

# Verify licenses
valid, message = lm.verify_license("premium@user.com", premium_key)
print(message)

Valid until 2028-09-11 (1095 days remaining)


In [6]:
from marearts_crystal import ma_crystal

class PasswordVault:
    def __init__(self, master_password):
        self.mask = ma_crystal(master_password)
    
    def store_password(self, service, username, password):
        data = f"{service}|{username}|{password}"
        return self.mask.encrypt_string(data)
    
    def get_password(self, encrypted_data):
        decrypted = self.mask.decrypt_string(encrypted_data)
        if decrypted:
            service, username, password = decrypted.split("|")
            return {"service": service, "username": username, "password": password}
        return None

# Usage
vault = PasswordVault("master-password-123")

# Store credentials
encrypted = vault.store_password("GitHub", "john_doe", "ghp_secrettoken123")
print(f"Encrypted: {encrypted[:50]}...")

# Retrieve credentials
credentials = vault.get_password(encrypted)
print(f"Service: {credentials['service']}")
print(f"Username: {credentials['username']}")

Encrypted: gAAAAABowsP44rawn3hneGIkVMAEGiDm9Vml1cPVf9CgDQ3w5K...
Service: GitHub
Username: john_doe
